In [ ]:
import mne

**read eeg data**

In [ ]:
path ='../data/ds005620/sub-1017/eeg/sub-1017_task-awake_acq-EC_eeg.vhdr'  #select any .vhdr file
data=mne.io.read_raw_brainvision(path, preload=True, verbose=True)


**data inspection**

In [ ]:
display(data)  #view metadata

In [ ]:
data.ch_names

In [ ]:
data.info['bads']   #check bad channels

In [ ]:
fig = data.plot(duration=10, show_scrollbars=False, scalings='auto', title='Raw EEG Data')

**preprocessing**

In [ ]:
ch=['Fz','Cz','Pz','P3','P4','Oz']   # select 6 channels of interest

In [ ]:
#re-referencing
data_processed = data.copy().set_eeg_reference(ref_channels="average")

for title, d in zip(["Before re-referencing", "After re-referencing"], [data, data_processed]):
    fig = d.copy().pick(picks=ch).plot(duration=10, show_scrollbars=False)
    fig.suptitle(title, size="xx-large", weight="bold")

In [ ]:
#low pass & high pass filtering
data_filtered = data_processed.copy().filter(l_freq=0.1, h_freq=40)
# plot power spectral density before and after filtering
for title, data in zip(["Before filtering", "After  filtering"], [data_processed, data_filtered]):
    fig = data.copy().pick(picks=ch).compute_psd(fmax=data.info['sfreq']/2).plot(exclude="bads")
    fig.suptitle(title, size="xx-large", weight="bold")
# Plot eeg signal before and after filtering in time domain
for title, data in zip(["Before  filtering", "After  filtering"], [data_processed, data_filtered]):
    fig = data.copy().pick(picks=ch).plot()
    fig.suptitle(title, size="xx-large", weight="bold")

print(f'The filtered data have the frequency range = {data_filtered.info["highpass"]:.02f} Hz to {data_filtered.info["lowpass"]:.02f} Hz')

In [ ]:
#ica

ica = mne.preprocessing.ICA(n_components=15, random_state=97, max_iter='auto')
ica.fit(data_filtered)


In [ ]:
_ = ica.plot_sources(data_filtered, show_scrollbars=False)

**spectral bands**

In [ ]:
alpha=data_filtered.copy().pick(picks=ch).filter(l_freq=8,h_freq=12)
_=alpha.plot()

In [ ]:
beta=data_filtered.copy().pick(picks=ch).filter(l_freq=12,h_freq=30)
_=beta.plot()

In [ ]:
theta=data_filtered.copy().pick(picks=ch).filter(l_freq=4,h_freq=8)
_=theta.plot()

In [ ]:
delta=data_filtered.copy().pick(picks=ch).filter(l_freq=1,h_freq=4)
_=delta.plot()